# RadHarmony: Evaluating a Foundation Model - Part 3




> **Please select `RadHarmony-(chexagent)` as the Python kernel.**

In [ ]:
import os
import pandas as pd

GPU_ID = 2
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
DEVICE = "cuda"

### Try it yourself: a different VQA model

Swap the recipe for **CheXagent-2** (`make_chexagent_vqa`), a CXR-specialist VLM.
The harness, dataset, prompt handling, and metrics stay put; only the two lines that
build the model change.

In [ ]:
from radharmony.evaluator import VQAEvaluator
from radharmony.dataset import VQARadDataset
from radharmony.evaluator.backbones import make_chexagent_vqa

VQA_RAD_DIR = "/mnt/NAS4/datasets/external/VQA-RAD/VQA_RAD Image Folder"

# CheXagent-2 instead of MedGemma; returns (transform, answerer) like the MedGemma recipe.
transform, answerer = make_chexagent_vqa(device=DEVICE)

vqa_ds = VQARadDataset(base_image_dir=VQA_RAD_DIR, transform=transform, cache_dir="/mnt/NAS4/projects/fli40_sharing/datathon26_summer_school/cache_chexagent_vqa")

# FILL IN VQAEvaluator's first (positional) argument.
#   docs: https://f10409.github.io/RadHarmony/evaluator/vqa.html
ev = VQAEvaluator(..., dataset=vqa_ds, output_dir="outputs/vqa_chexagent")
ev.evaluate()

### Try it yourself: a different report generator

`make_chexagent_generator` (CheXagent-2) and `make_maira2_generator` (MAIRA-2) both
return `(transform, generator)` and drop straight into `ReportGenerationEvaluator`.
Swap the recipe; keep the MIMIC-CXR dataset and `findings` scoring.

> **Two environments.** CheXagent-2 and MAIRA-2 pin an old `transformers`, which
> conflicts with the newer one RadGraph (`radeval`) needs, so the `chexagent`
> kernel has no `radeval`. Unlike MedGemma (whose kernel bundles both), these
> models cannot generate and score in one `evaluate()` call. Split it in two:
>
> - **Stage A (this `RadHarmony-(chexagent)` kernel):** generate the reports and
>   write the (reference, generated) pairs with `ev.generate_only(...)`.
> - **Stage B (switch to `RadHarmony-(medgemma)`):** load that parquet and score
>   it with RadGraph. No CheXagent, no transformers conflict.

In [ ]:
# Stage A (RadHarmony-(chexagent) kernel): generate only, no scoring.
from radharmony.evaluator.backbones import make_chexagent_generator
from radharmony.dataset import MIMICCXRDataset
from radharmony.evaluator import ReportGenerationEvaluator

MIMIC_IMAGE_DIR = "/mnt/NAS4/datasets/external/MIMIC-CXR-V2-AWS/files/"
MIMIC_REPORT_CSV = "/mnt/NAS4/datasets/external/MIMIC-CXR-V2-AWS/cxr-study-list.csv.gz"

# from radharmony.evaluator.backbones import make_maira2_generator  # MAIRA-2 instead
transform, generator = make_chexagent_generator(device=DEVICE)

df_harmonized = pd.read_parquet("data/df_mimic_harmonized.parquet")
gen_ds = MIMICCXRDataset(
    base_image_dir=MIMIC_IMAGE_DIR,
    report_csv_path=MIMIC_REPORT_CSV,
    transform=transform,
    output_report=True,
    cache_dir="/mnt/NAS4/projects/fli40_sharing/datathon26_summer_school/cache_chexagent_gen",
    harmonized_df=df_harmonized,
)

# FILL IN ref_section. metrics move to Stage B, so they are omitted here.
#   docs: https://f10409.github.io/RadHarmony/evaluator/report_generation.html
ev = ReportGenerationEvaluator(
    generator,
    dataset=gen_ds,
    ref_section=...,  # which report section to score?
    max_samples=1000,  # small cap for the demo; remove for a full run
)

# generate_only writes a parquet of (sample_id, reference, hypothesis) pairs and
# does NOT import radeval, so it runs cleanly in the chexagent kernel. Score it in
# Stage B below.
PAIRS_PARQUET = "outputs/chexagent_pairs.parquet"
ev.generate_only(PAIRS_PARQUET)
print(f"wrote pairs -> {PAIRS_PARQUET}")

In [ ]:
from viz_helpers import show_reports

show_reports(gen_ds, generator)

### Stage B: score the reports (RadGraph)

Now **switch the Python kernel to `RadHarmony-(medgemma)`** (the one with
`radeval` installed) and run the cell below. It reads the parquet Stage A wrote
and scores the (reference, generated) pairs with RadGraph F1: the same numbers an
inline `evaluate()` would produce, just computed in the environment that can import
RadGraph. Nothing above this point needs to rerun.

In [ ]:
# Stage B (switch the kernel to RadHarmony-(medgemma), which has radeval).
# Only pandas + the metric function are needed here; no model, no CheXagent.
import pandas as pd
from radharmony.evaluator.metrics.language import compute_report_metrics

pairs = pd.read_parquet("outputs/chexagent_pairs.parquet")

# reference = findings parsed in Stage A; hypothesis = CheXagent's generated report.
# FILL IN metrics.
#   docs: https://f10409.github.io/RadHarmony/evaluator/report_generation.html
scores = compute_report_metrics(
    refs=pairs["reference"].tolist(),
    hyps=pairs["hypothesis"].tolist(),
    metrics=...,  # n-gram overlap + RadGraph F1
)
pd.DataFrame([{"label": "report", **scores}])

## Summary

You swapped in CheXagent-2 (optionally MAIRA-2) and scored its reports with the same RadGraph metrics, using the two-stage generate-then-score split across kernels. That rounds out the evaluation tour across classification, segmentation, VQA, and report generation.